In [1]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
import zipfile
from tqdm import tqdm  # progress bar

In [2]:
# === Upload ZIP dataset ===
print("Silakan upload file ZIP dataset retina...")
uploaded = files.upload()
zip_path = next(iter(uploaded))  # ambil nama file pertama

Silakan upload file ZIP dataset retina...


Saving images_training.zip to images_training.zip


In [3]:
# Ekstrak ke folder "training_image"
dataset_folder = "training_image"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_folder)

In [4]:
# Buat folder output
output_folder = "enhanced_training_image"
os.makedirs(output_folder, exist_ok=True)

In [5]:
# === Fungsi enhance khusus retina vessel ===
def enhance_retina(img_gray, clip_limit=3.0, grid_size=(8,8), alpha=1.5, beta=-0.5):
    """Enhance citra retina dengan CLAHE + Unsharp Masking"""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    clahe_img = clahe.apply(img_gray)

    # Gaussian blur untuk unsharp masking
    blur = cv2.GaussianBlur(clahe_img, (5,5), 0)
    sharpened = cv2.addWeighted(clahe_img, alpha, blur, beta, 0)

    return clahe_img, sharpened

In [9]:
# === Loop semua gambar dalam dataset ===
valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.tif')
image_files = [f for f in os.listdir(dataset_folder) if f.lower().endswith(valid_ext)]

for file in tqdm(image_files, desc="Enhancing images"):
    path = os.path.join(dataset_folder, file)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    clahe_img, sharpened = enhance_retina(img)

    # Simpan hasil
    cv2.imwrite(os.path.join(output_folder, f"clahe_{file}"), clahe_img)
    cv2.imwrite(os.path.join(output_folder, f"sharpened_{file}"), sharpened)

Enhancing images: 0it [00:00, ?it/s]
